<a href="https://colab.research.google.com/github/victorbaraunaAcad/crise-saude-ufam/blob/Leonardo-branch/Tratamento_e_integracao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Configuração Inicial e Importação de Bibliotecas

In [ ]:
import json, time, datetime, pathlib, unicodedata
from getpass import getpass

import requests
import pandas as pd

print(f"pandas  {pd.__version__}")
print(f"requests {requests.__version__}")

pandas  2.2.3
requests 2.32.4


# Autenticação e Clonagem do Repositório GitHub

In [ ]:
%%script echo "Ignorando comandos de Git"
from getpass import getpass

token = getpass("Cole o token do GitHub: ")

%cd /content

!git clone -b Leonardo-branch https://{token}@github.com/victorbaraunaAcad/crise-saude-ufam.git
%cd crise-saude-ufam

!git config user.name "8january"
!git config user.email "leonardobrandaoamarante@gmail.com"

Cole o token do GitHub: ··········
/content
Cloning into 'crise-saude-ufam'...
remote: Enumerating objects: 388, done.
remote: Counting objects: 100% (223/223), done.
remote: Compressing objects: 100% (194/194), done.
remote: Total 388 (delta 63), reused 182 (delta 28), pack-reused 165 (from 1)
Receiving objects: 100% (388/388), 192.84 MiB | 27.34 MiB/s, done.
Resolving deltas: 100% (102/102), done.
/content/crise-saude-ufam


In [ ]:
%%script echo "Ignorando comandos de Git"
%cd /content/crise-saude-ufam

# Apaga o registro do submódulo/pasta duplicada do rastreamento do Git
!git rm --cached crise-saude-ufam -f 2>/dev/null || true

# Envia a correção para a sua branch no GitHub
!git commit -m "Remove pasta duplicada e referencia de submodulo"
!git push origin Leonardo-branch

/content/crise-saude-ufam
rm 'crise-saude-ufam'
[Leonardo-branch af27455] Remove pasta duplicada e referencia de submodulo
 1 file changed, 1 deletion(-)
 delete mode 160000 crise-saude-ufam
Enumerating objects: 3, done.
Counting objects: 100% (3/3), done.
Delta compression using up to 2 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (2/2), 257 bytes | 257.00 KiB/s, done.
Total 2 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/victorbaraunaAcad/crise-saude-ufam.git
   eb05442..af27455  Leonardo-branch -> Leonardo-branch


# Configuração de Diretórios do Projeto

In [ ]:
RAIZ     = pathlib.Path.cwd()
BRUTOS   = RAIZ / "dados_brutos"
TRATADOS = RAIZ / "dados_tratados"
BRUTOS.mkdir(exist_ok=True); TRATADOS.mkdir(exist_ok=True)
print("Trabalhando em:", RAIZ)

Trabalhando em: /content/crise-saude-ufam


In [ ]:
%%script echo "Ignorando comandos de Git"
%cd /content/crise-saude-ufam
!git remote set-url origin https://github.com/victorbaraunaAcad/crise-saude-ufam.git

/content/crise-saude-ufam


# Configuração de Remoto do Git e Git LFS

In [ ]:
%%script echo "Ignorando comandos de Git"
!apt-get update && apt-get install git-lfs -y

%cd /content/crise-saude-ufam
!git lfs install
!git lfs pull
!git pull

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:5 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git-lfs is already the newest version (3.4.1-1ubuntu0.4).
0 upgraded, 0 newly installed, 0 to remove and 9 not upgraded.
/content/crise-saude-ufam
Updated Git hooks.
Git 

# Função de Normalização de Texto

In [ ]:
def normalizar(s):
    if pd.isna(s): return ""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return s.upper().strip()

## 1. Mapeamento Territorial e Normalização (IBGE)

### Objetivo
Construir a tabela mestre de municípios brasileiros com padronização de identificadores territoriais e normalização textual para cruzamentos futuros.

### Decisões Técnicas e Metodológicas

* **Compatibilização de Códigos IBGE (7 vs 6 Dígitos):**
  * O IBGE disponibiliza códigos territoriais com **7 dígitos** (incluindo o dígito verificador final), enquanto os sistemas do DATASUS (SIH/SIM) utilizam a versão truncada de **6 dígitos**.
  * **Decisão:** Mantiveram-se ambas as chaves (`codigo_ibge` como `int64` para cruzamentos socioeconômicos e `codigo_ibge_6` gerado via divisão inteira por 10 `// 10` para integração com o DATASUS).

* **Tratamento e Normalização de Texto (`normalizar`):**
  * **Problema:** Nomes de municípios contêm acentuações, caracteres especiais e variações de caixa (ex: "Santarém" vs "SANTAREM"), o que inviabiliza *joins* por texto.
  * **Decisão:** A função remove diacríticos através da decomposição unicode `NFKD`, descarta caracteres não-ASCII, converte a string para maiúsculas e remove espaços sobressalentes nas extremidades.

In [ ]:
dfs_loc = []
for arq in BRUTOS.glob("ibge_localidades_*.json"):
    dados = json.loads(arq.read_text(encoding="utf-8"))
    dfs_loc.append(pd.json_normalize(dados))

df_mun = pd.concat(dfs_loc, ignore_index=True)
df_mun = df_mun.rename(columns={
    "id": "codigo_ibge",
    "nome": "nome_municipio",
    "microrregiao.mesorregiao.UF.sigla": "uf",
    "microrregiao.mesorregiao.UF.id": "uf_codigo"
})[["codigo_ibge", "nome_municipio", "uf", "uf_codigo"]].drop_duplicates("codigo_ibge")

df_mun["codigo_ibge"] = df_mun["codigo_ibge"].astype("int64")
df_mun["codigo_ibge_6"] = df_mun["codigo_ibge"].astype(str).str[:6].astype("int64")
df_mun["nome_norm"] = df_mun["nome_municipio"].map(normalizar)

## 2. Integração de Indicadores Socioeconômicos (SIDRA e PNAD)

### Objetivo
Extrair e agregar os dados de População Residente, PIB Total e Taxa de Desocupação (desemprego) por Estado/Município.

### Decisões Técnicas e Metodológicas

* **Extração de JSONs Hierárquicos (`extrair_sidra`):**
  * A API do SIDRA retorna estruturas aninhadas (`resultados -> series -> localidade/serie`). A função navega recursivamente pela árvore mantendo o mapeamento com o código territorial e convertendo valores nulos ou inválidos para tipo numérico seguro via `pd.to_numeric(..., errors="coerce")`.

* **Agregação do Desemprego por UF:**
  * Como a Taxa de Desocupação da PNAD Contínua é mensurada em nível estadual, os dados foram agregados pela média do período por `uf_codigo`. A associação com os municípios ocorre via *Left Join* na tabela mestre.

In [ ]:
def extrair_sidra(padrao, nome_coluna):
    registros = []
    for arq in BRUTOS.glob(padrao):
        dados = json.loads(arq.read_text(encoding="utf-8"))
        for item in dados:
            for res in item.get("resultados", []):
                for serie in res.get("series", []):
                    cod_mun = int(serie["localidade"]["id"])
                    valor = list(serie["serie"].values())[0]
                    registros.append({"codigo_ibge": cod_mun, nome_coluna: valor})
    df = pd.DataFrame(registros).drop_duplicates("codigo_ibge")
    df[nome_coluna] = pd.to_numeric(df[nome_coluna], errors="coerce")
    return df

df_pop = extrair_sidra("ibge_populacao_*.json", "populacao")
df_pib = extrair_sidra("ibge_pib_total_*.json", "pib_total")

In [ ]:
arq_desemp = BRUTOS / "ibge_desemprego_ufs.json"
dados_desemp = json.loads(arq_desemp.read_text(encoding="utf-8"))
linhas_desemp = []
for item in dados_desemp:
    for res in item.get("resultados", []):
        for serie in res.get("series", []):
            for val in serie["serie"].values():
                linhas_desemp.append({
                    "uf_codigo": int(serie["localidade"]["id"]),
                    "taxa_desocupacao": pd.to_numeric(val, errors="coerce")
                })
df_desemp_uf = pd.DataFrame(linhas_desemp).groupby("uf_codigo")["taxa_desocupacao"].mean().reset_index()

print(f"✓ IBGE processado: {len(df_mun)} municípios mapeados.")

✓ IBGE processado: 948 municípios mapeados.


## 3. Processamento de Microdados de Saúde (DATASUS - SIH e SIM)

### Objetivo
Processar arquivos `.parquet` contendo registros de internações (SIH - Sistema de Informações Hospitalares) e mortalidade (SIM - Sistema de Informações sobre Mortalidade).

### Decisões Técnicas e Metodológicas

* **Leitura Seletiva de Colunas e Tolerância a Falhas:**
  * Devido ao elevado volume de dados, foram carregadas estritamente as colunas necessárias (`MUNIC_RES` e `VAL_TOT` no SIH; `CODMUNRES` no SIM), reduzindo o consumo de memória RAM.
  * O bloco de leitura utiliza tratativa de exceção (`try-except`) para ignorar arquivos com esquemas corrompidos sem interromper o fluxo de execução.

* **Agregação no Nível Municipal (6 Dígitos):**
  * Agrupamento por código de município de residência do paciente para calcular:
    1. `total_internacoes`: Contagem bruta de AIHs processadas.
    2. `custo_total_internacoes`: Soma dos valores totais do processamento hospitalar (`VAL_TOT`).
    3. `total_obitos_sim`: Contagem de declarações de óbito registradas.

In [ ]:
arqs_sih = list(BRUTOS.glob("RD*.parquet")) + list(BRUTOS.glob("sih_*.parquet"))
dfs_sih = []

for arq in arqs_sih:
    try:
        df_temp = pd.read_parquet(arq, columns=["MUNIC_RES", "VAL_TOT"])
        dfs_sih.append(df_temp)
    except Exception:
        pass  # Ignora arquivos com esquemas divergentes

if dfs_sih:
    df_sih_raw = pd.concat(dfs_sih, ignore_index=True)
    df_sih_raw["codigo_ibge_6"] = pd.to_numeric(df_sih_raw["MUNIC_RES"], errors="coerce")
    df_sih_raw["VAL_TOT"] = pd.to_numeric(df_sih_raw["VAL_TOT"], errors="coerce")

    df_sih_agg = df_sih_raw.groupby("codigo_ibge_6").agg(
        total_internacoes=("MUNIC_RES", "count"),
        custo_total_internacoes=("VAL_TOT", "sum")
    ).reset_index()
else:
    df_sih_agg = pd.DataFrame(columns=["codigo_ibge_6", "total_internacoes", "custo_total_internacoes"])

In [ ]:
arqs_sim = list(BRUTOS.glob("DO*.parquet"))
dfs_sim = []

for arq in arqs_sim:
    try:
        df_temp = pd.read_parquet(arq, columns=["CODMUNRES"])
        dfs_sim.append(df_temp)
    except Exception:
        pass

if dfs_sim:
    df_sim_raw = pd.concat(dfs_sim, ignore_index=True)
    df_sim_raw["codigo_ibge_6"] = pd.to_numeric(df_sim_raw["CODMUNRES"], errors="coerce")

    df_sim_agg = df_sim_raw.groupby("codigo_ibge_6").agg(
        total_obitos_sim=("CODMUNRES", "count")
    ).reset_index()
else:
    df_sim_agg = pd.DataFrame(columns=["codigo_ibge_6", "total_obitos_sim"])

print("✓ DATASUS consolidado:")
print(f"  - Registros agregados do SIH: {len(df_sih_agg)}")
print(f"  - Registros agregados do SIM: {len(df_sim_agg)}")

✓ DATASUS consolidado:
  - Registros agregados do SIH: 3024
  - Registros agregados do SIM: 1031


## 4. Consolidação da Base Final e Engenharia de Atributos

### Objetivo
Executar a fusão (*merge*) de todas as fontes de dados em uma única matriz municipal e calcular indicadores derivados ajustados.

### Decisões Técnicas e Metodológicas

* **Estratégia de Junção (`Left Join` na Tabela Mestre):**
  * Utiliza-se `df_mun` como base (*left*), garantindo a preservação de todos os municípios do Brasil, mesmo aqueles sem registros de internações/óbitos no período analisado.

* **Alinhamento Estrito de Tipos de Dados:**
  * Forçamento explícito do tipo `int64` para todas as chaves de junção (`codigo_ibge`, `codigo_ibge_6`, `uf_codigo`). Isso previne falhas sintáticas e garante que o Pandas não converta chaves inteiras em *floats* devido à presença de `NaN`.

* **Tratamento de Valores Ausentes (`fillna(0)`):**
  * Municípios sem registros no SIH ou SIM recebem valor `0` nas colunas de contagem e custo, visto que a ausência de registro nas tabelas do DATASUS equivale a zero ocorrência notificada.

* **Proteção Contra Divisão por Zero (`pop_segura`):**
  * **Problema:** Municípios recém-criados ou com dados demográficos inconsistentes podem apresentar população igual a 0. A divisão direta geraria valores `inf` (infinito).
  * **Decisão:** Substituição de `0` por `np.nan` na variável temporária `pop_segura`. Assim, divisões por zero resultam em `NaN`, preservando a integridade estatística da base.

* **Ajuste de Escala do PIB per Capita:**
  * O PIB divulgado pelo IBGE na tabela SIDRA é expresso em **milhares de Reais ($R\$ 1.000,00$)**. O cálculo do PIB *per capita* multiplica o valor por 1.000 antes da divisão populacional para expressar o resultado final em moeda corrente ($R\$/habitante$).

In [ ]:
import numpy as np

# 1. Alinhamento de tipos e derivação explícita da chave de 6 dígitos
df_pop["codigo_ibge"] = df_pop["codigo_ibge"].astype("int64")
df_pib["codigo_ibge"] = df_pib["codigo_ibge"].astype("int64")
df_desemp_uf["uf_codigo"] = df_desemp_uf["uf_codigo"].astype("int64")
df_mun["codigo_ibge"] = df_mun["codigo_ibge"].astype("int64")

# Extrai e garante a chave de 6 dígitos em todas as tabelas envolvidas
df_mun["codigo_ibge_6"] = (df_mun["codigo_ibge"] // 10).astype("int64")
df_sih_agg["codigo_ibge_6"] = df_sih_agg["codigo_ibge_6"].astype("int64")
df_sim_agg["codigo_ibge_6"] = df_sim_agg["codigo_ibge_6"].astype("int64")

# 2. Merges sequenciais na base consolidada
base_final = df_mun.merge(df_pop, on="codigo_ibge", how="left")
base_final = base_final.merge(df_pib, on="codigo_ibge", how="left")
base_final = base_final.merge(df_desemp_uf, on="uf_codigo", how="left")
base_final = base_final.merge(df_sih_agg, on="codigo_ibge_6", how="left")
base_final = base_final.merge(df_sim_agg, on="codigo_ibge_6", how="left")

# 3. Preenchimento de ausentes nas métricas de contagem do DATASUS
cols_metricas = ["total_internacoes", "custo_total_internacoes", "total_obitos_sim"]
for col in cols_metricas:
    if col in base_final.columns:
        base_final[col] = base_final[col].fillna(0)

# 4. Atributos derivados com tratamento de divisão por zero
pop_segura = base_final["populacao"].replace(0, np.nan)

base_final["pib_per_capita_calculado"] = (base_final["pib_total"] * 1000) / pop_segura
base_final["taxa_internacao_por_10k"] = (base_final["total_internacoes"] / pop_segura) * 10000

# 5. Exportação dos arquivos finais
arq_csv = TRATADOS / "base_consolidada_municipios.csv"
arq_parquet = TRATADOS / "base_consolidada_municipios.parquet"

base_final.to_csv(arq_csv, index=False, encoding="utf-8-sig")
base_final.to_parquet(arq_parquet, index=False)

print(f"✓ Base consolidada criada ({base_final.shape[0]} linhas, {base_final.shape[1]} colunas).")
print(f"✓ Salvo em: {arq_parquet.relative_to(RAIZ)}")
base_final.head()

✓ Base consolidada criada (948 linhas, 14 colunas).
✓ Salvo em: dados_tratados/base_consolidada_municipios.parquet


,codigo_ibge,nome_municipio,uf,uf_codigo,codigo_ibge_6,nome_norm,populacao,pib_total,taxa_desocupacao,total_internacoes,custo_total_internacoes,total_obitos_sim,pib_per_capita_calculado,taxa_internacao_por_10k
0,1300029,Alvarães,AM,13.0,130002,ALVARAES,16938.0,217333.0,9.1125,3454.0,1773599.08,109.0,12831.089857,2039.201795
1,1300060,Amaturá,AM,13.0,130006,AMATURA,11633.0,118872.0,9.1125,1730.0,849855.94,66.0,10218.516290,1487.148629
2,1300086,Anamã,AM,13.0,130008,ANAMA,9981.0,138599.0,9.1125,5734.0,2160800.84,65.0,13886.283939,5744.915339
3,1300102,Anori,AM,13.0,130010,ANORI,18135.0,248435.0,9.1125,6648.0,3329233.82,118.0,13699.200441,3665.839537
4,1300144,Apuí,AM,13.0,130014,APUI,21971.0,297810.0,9.1125,3472.0,2041115.00,171.0,13554.685722,1580.264895


## 5. Cruzamento com Indicadores de Eventos Climáticos Extremos

### Objetivo
Relacionar os dados de base municipal (socioeconômicos e de saúde) com os indicadores coletados via scraping durante os desastres ambientais estudados.

### Decisões Técnicas e Metodológicas

* **Segmentação por Recorte Geográfico do Evento:**
  * **Rio Grande do Sul (RS):** Foco no impacto de enchentes severas.
  * **Amazonas (AM):** Foco na estiagem e secas dos rios.
  * **Centro-Oeste (CO):** Foco na exposição a ondas de calor e queimadas.

* **Estratégia de Agregação:**
  * Os dados de scraping capturam o ápice do desastre. Para evitar distorções com médias temporais suavizadas, extraem-se os valores de pico (`max()`) de pessoas afetadas, desalojadas e óbitos diretamente relacionados ao evento.

In [ ]:
import pandas as pd

# 1. Carregar a base geral de municípios (com dados IBGE + DATASUS)
df_muni = pd.read_csv("dados_tratados/base_consolidada_municipios.csv")

# 2. Carregar as tabelas de indicadores do scraping
df_rs_scraping = pd.read_csv("dados_tratados/base_enchentes_indicadores_rs.csv")
df_am_scraping = pd.read_csv("dados_tratados/base_estiagem_indicadores_am.csv")
df_co_scraping = pd.read_csv("dados_tratados/base_calor_indicadores_co.csv")

# BASE 1: ENCHENTES (Rio Grande do Sul)
df_rs = df_muni[df_muni['uf'] == 'RS'].copy()

# Agregar picos/resumo do scraping do RS
resumo_rs = {
    'pico_obitos_rs': df_rs_scraping['obitos'].max(),
    'pico_desalojados_rs': df_rs_scraping['desalojados'].max(),
    'pico_pessoas_afetadas_rs': df_rs_scraping['pessoas_afetadas'].max(),
    'max_nivel_guaiba_m': df_rs_scraping['nivel_guaiba_m'].max(),
    'meses_em_crise': df_rs_scraping['em_crise'].notna().sum()
}
for col, val in resumo_rs.items():
    df_rs[col] = val

df_rs.to_csv("dados_tratados/base_enchentes_indicadores_rs_final.csv", index=False)
df_rs.to_parquet("dados_tratados/base_enchentes_indicadores_rs_final.parquet", index=False)


# BASE 2: ESTIAGEM (Amazonas)
df_am = df_muni[df_muni['uf'] == 'AM'].copy()

# Agregar picos/resumo do scraping do AM
resumo_am = {
    'max_municipios_emergencia': df_am_scraping['municipios_emergencia'].max(),
    'pico_pessoas_afetadas_mil': df_am_scraping['pessoas_afetadas_mil'].max(),
    'min_cota_rio_negro_m': df_am_scraping['cota_rio_negro_m'].min(),
    'meses_em_crise': df_am_scraping['em_crise'].notna().sum()
}
for col, val in resumo_am.items():
    df_am[col] = val

df_am.to_csv("dados_tratados/base_estiagem_indicadores_am_final.csv", index=False)
df_am.to_parquet("dados_tratados/base_estiagem_indicadores_am_final.parquet", index=False)


# BASE 3: ONDA DE CALOR (Centro-Oeste)
df_co = df_muni[df_muni['uf'].isin(['MT', 'MS', 'GO', 'DF'])].copy()

# Agregar picos/resumo do scraping do CO
resumo_co = {
    'temp_max_cuiaba_c': df_co_scraping['temp_max_cuiaba_c'].max(),
    'temp_max_brasil_c': df_co_scraping['temp_max_brasil_c'].max(),
    'umidade_minima_pct': df_co_scraping['umidade_minima_pct'].min(),
    'max_dias_consecutivos_40c': df_co_scraping['dias_consecutivos_40c'].max(),
    'max_focos_calor_mt': df_co_scraping['focos_calor_mt'].max(),
    'meses_em_crise': df_co_scraping['em_crise'].notna().sum()
}
for col, val in resumo_co.items():
    df_co[col] = val

df_co.to_csv("dados_tratados/base_calor_indicadores_co_final.csv", index=False)
df_co.to_parquet("dados_tratados/base_calor_indicadores_co_final.parquet", index=False)

print("As 3 bases temáticas consolidadas foram geradas com sucesso em 'dados_tratados/'!")

As 3 bases temáticas consolidadas foram geradas com sucesso em 'dados_tratados/'!


In [ ]:
%%script echo "Ignorando comandos de Git"
!git status

On branch Leonardo-branch
Your branch is up to date with 'origin/Leonardo-branch'.

nothing to commit, working tree clean


In [ ]:
%%script echo "Ignorando comandos de Git"
TOKEN = getpass("Cole o token do GitHub (fine-grained, Contents: Read/write): ")
USUARIO = "victorbaraunaAcad"
REPO = "crise-saude-ufam"

!git remote set-url origin https://{TOKEN}@github.com/{USUARIO}/{REPO}.git
!git checkout Leonardo-branch

!git push -u origin Leonardo-branch

Cole o token do GitHub (fine-grained, Contents: Read/write): ··········
Already on 'Leonardo-branch'
Your branch is up to date with 'origin/Leonardo-branch'.
branch 'Leonardo-branch' set up to track 'origin/Leonardo-branch'.
Everything up-to-date
